# 🧠 GenAI Task 1: Advanced Prompt Engineering with LangChain
---
### 🚀 Architectural Overview
In production-grade Generative AI, hardcoded strings are a primary source of technical debt and system fragility. This notebook explores the transition from static text to **Dynamic Prompt Architectures**. By leveraging the LangChain framework, we implement a modular approach that separates the **Prompt Logic** (the structure) from the **Input Data** (the context).

**Core Objectives:**
*   **Abstraction:** Decouple prompt definitions from application code.
*   **Validation:** Implement defensive layers to sanitize LLM inputs.
*   **Polymorphism:** Swap prompt templates dynamically based on user intent or system state.
*   **Scaling:** Efficiently manage high-throughput data streams through batch interpolation.

## 🛠️ Section 1: Environment Setup
---
We begin by ensuring the necessary LangChain dependencies are installed. This provides the foundation for our prompt engineering tasks.

In [ ]:
!pip install langchain

## 📦 Section 2: Core Library Imports
---
We import the essential components from `langchain_core`. Using the core package is the recommended practice for modern LangChain applications to ensure stability and modularity.

In [ ]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field, field_validator
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnableParallel

### 🎯 Task 1: Decoupling Logic with `PromptTemplate`

We initiate the process by implementing `PromptTemplate` to achieve **Template Decoupling**. This allows us to define a fixed semantic structure while exposing specific slots for dynamic **String Interpolation**. This ensures that the core 'Instruction Set' remains immutable even as the 'Context Data' changes.

In [ ]:
# Task 1: Defining a basic PromptTemplate with a single input variable
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners"
)

# Format the template by passing the 'topic' value
print(template.format(topic="AI"))

### 🔄 Task 2: Multi-Variable Contextualization

Scalable LLM applications require multi-dimensional context mapping. Here, we architect templates that ingest `topic`, `audience`, and `tone` as independent state variables. This enables a single logic block to generate specialized outputs tailored to specific user personas, from highly technical engineers to elementary students.

In [ ]:
# Task 2: Creating a complex template that accepts multiple dynamic inputs
multi_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

# Demonstrate formatting with different combinations of inputs
print(multi_template.format(topic="AI", audience="beginners", tone="friendly"))
print(multi_template.format(topic="Python", audience="kids", tone="fun"))
print(multi_template.format(topic="Deep Learning", audience="engineers", tone="technical"))

### 🎭 Task 3: Template Specialization & Functional Polymorphism

By defining specialized templates—**Pedagogical (Teaching)**, **Evaluative (Interview)**, and **Narrative (Story)**—we demonstrate **Functional Polymorphism**. We show how to swap the underlying 'flavor' of the prompt while maintaining a consistent data schema, allowing for rapid A/B testing and persona switching.

In [ ]:
# Task 3: Defining specialized templates for different use cases (Teaching, Interview, Story)
teaching_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} clearly step by step"
)

interview_template = PromptTemplate(
    input_variables=["topic"],
    template="Ask 3 interview questions about {topic}"
)

story_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} as a story"
)

# Output the formatted prompts for a specific subject
print(teaching_template.format(topic="Machine Learning"))
print(interview_template.format(topic="Machine Learning"))
print(story_template.format(topic="Machine Learning"))

### 💬 Task 4: Chat-Centric Architectures (`ChatPromptTemplate`)

To optimize for Chat-Tuned Models (like GPT-4 or Gemini), we utilize `ChatPromptTemplate`. This allows us to manage **Stateful Roles** by explicitly defining `system` (Behavioral Persona) and `human` (User Intent) messages, providing the model with a clear conversational hierarchy.

In [ ]:
# Task 4: Utilizing ChatPromptTemplate to structure messages for Chat Models
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful {role}"),
    ("user", "Explain {topic}")
])

# Generate a list of message objects formatted with the provided variables
messages = chat_template.format_messages(role="teacher", topic="Neural Networks")
print(messages)

### 🛡️ Task 5: Defensive Prompting & Input Validation Layers

To ensure system reliability and mitigate 'Garbage-In, Garbage-Out' (GIGO) scenarios, we implement a **Defensive Validation Layer**. This logic acts as a firewall, sanitizing input variables like `audience` and `tone` against a predefined whitelist before they reach the inference stage.

In [ ]:
class GenerationConfig(BaseModel):
    """Advanced configuration schema using Pydantic V2 validation."""
    topic: str = Field(..., min_length=2)
    audience: Literal["beginner", "intermediate", "expert"] = "beginner"
    tone: Literal["formal", "casual", "fun"] = "formal"
    style: Optional[str] = "teaching"

    @field_validator("topic")
    @classmethod
    def topic_must_not_be_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("The 'topic' field cannot be empty or just whitespace.")
        return v.strip()

### 🏭 Task 6: The Modular Prompt Factory

This section implements the **Prompt Factory Pattern**. By integrating conditional logic with our validation layers, we create a system that dynamically selects and constructs the most effective prompt object based on the requested `style`, showcasing a mature approach to prompt management.

In [ ]:
class AdvancedPromptFactory:
    """Advanced factory for managing dynamic prompt architectures."""

    TEMPLATES = {
        "teaching": "Explain {topic} for a {audience} audience in a {tone} teaching style.",
        "interview": "Generate 3 technical interview questions regarding {topic} targeted at an {audience} level.",
        "storytelling": "Narrate the concept of {topic} for an {audience} using a {tone} storytelling approach."
    }

    @classmethod
    def get_prompt(cls, config: GenerationConfig) -> str:
        template_str = cls.TEMPLATES.get(config.style, "Explain {topic}")

        prompt = PromptTemplate.from_template(template_str)
        # Using LCEL compatible invocation
        return prompt.invoke({
            "topic": config.topic,
            "audience": config.audience,
            "tone": config.tone
        }).to_string()

# Example of advanced usage with validation
try:
    config = GenerationConfig(topic="Quantum Computing", audience="expert", tone="formal", style="interview")
    print(f"--- Advanced Generated Prompt ---\n{AdvancedPromptFactory.get_prompt(config)}")
except Exception as e:
    print(f"Validation Error: {e}")

### 🚀 Task 7: High-Throughput Batch Processing

We conclude by stress-testing our architecture via **Batch Interpolation**. This highlights the scalability of reusable components, showing how a single template can process a list of data points with zero code duplication and perfect consistency across the output stream.

In [ ]:
# Advanced Batch Processing using Parallel Runnables or mapping
topics_list = ["AI", "Python", "Data Science", "Deep Learning", "NLP"]

batch_template = PromptTemplate.from_template("Summarize {topic} in one sentence.")

# Efficiently process batch using LangChain's batch method
responses = batch_template.batch([{"topic": t} for t in topics_list])

for i, res in enumerate(responses):
    print(f"{topics_list[i]}: {res.to_string()}")